In [1]:
import hail as hl
# Initialize Hail 
hl.init(default_reference = 'GRCh38')

Running on Apache Spark version 3.1.3
SparkUI available at http://ibd-exome-qc-m.c.daly-ibd.internal:41899
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.99-57537fea08d4
LOGGING: writing to /home/hail/hail-20230418-0107-0.2.99-57537fea08d4.log


In [2]:
def pc_project(
        mt: hl.MatrixTable,
        loadings_ht: hl.Table,
        loading_location: str = "loadings",
        af_location: str = "pca_af"
) -> hl.Table:
    """
    Projects samples in `mt` on pre-computed PCs.
    :param MatrixTable mt: MT containing the samples to project
    :param Table loadings_ht: HT containing the PCA loadings and allele frequencies used for the PCA
    :param str loading_location: Location of expression for loadings in `loadings_ht`
    :param str af_location: Location of expression for allele frequency in `loadings_ht`
    :return: Table with scores calculated from loadings in column `scores`
    :rtype: Table
    """
    n_variants = loadings_ht.count()
    mt = mt.annotate_rows(
        pca_loadings=loadings_ht[mt.row_key][loading_location],
        pca_af=loadings_ht[mt.row_key][af_location]
    )
    mt = mt.filter_rows(hl.is_defined(mt.pca_loadings) & hl.is_defined(mt.pca_af) &
                        (mt.pca_af > 0) & (mt.pca_af < 1))
    gt_norm = (mt.GT.n_alt_alleles() - 2 * mt.pca_af) / hl.sqrt(n_variants * 2 * mt.pca_af * (1 - mt.pca_af))
    mt = mt.annotate_cols(scores=hl.agg.array_sum(mt.pca_loadings * gt_norm))
    return mt.cols().select('scores')

In [3]:
# mt = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round2/5.additional_sample_qc/merged_annotated.mt")


In [5]:
# gnomad_sites_ht = hl.read_table('gs://gcp-public-data--gnomad/release/3.1.2/ht/genomes/gnomad.genomes.v3.1.2.sites.ht')

# nfe_index = gnomad_sites_ht.freq_index_dict['nfe-adj'].collect()[0]
# mt = mt.annotate_rows(nfe = gnomad_sites_ht[mt.row_key].freq[nfe_index].AF)

# print("Filtering gnomAD NFE AF > 0.01...")
# mt = mt.filter_rows(mt.nfe > 0.01)
# mt.count()

Filtering gnomAD NFE AF > 0.01...


(472157, 199489)

In [6]:
# #Import 1000G
# print("Reading in 1KG data...")
# mt_1kg = hl.experimental.load_dataset(name='1000_Genomes_HighCov_autosomes', version='NYGC_30x_phased', reference_genome='GRCh38')
# mt_1kg = mt_1kg.annotate_rows(nfe = gnomad_sites_ht[mt_1kg.row_key].freq[nfe_index].AF)
# mt_1kg = mt_1kg.filter_rows(mt_1kg.nfe > 0.01)

# print("Merging datasets...")

# mt = mt.select_entries('GT')
# mt_1kg = mt_1kg.select_entries('GT')

# mt = mt.select_cols(mt.sample_qc.call_rate, mt.sample_qc.n_called, mt.sample_qc.n_not_called, 
#     mt.sample_qc.n_hom_ref, mt.sample_qc.n_het, mt.sample_qc.n_hom_var, mt.sample_qc.n_non_ref, mt.sample_qc.n_singleton, 
#     mt.sample_qc.n_snp, mt.sample_qc.n_insertion, mt.sample_qc.n_deletion, mt.sample_qc.n_transition, mt.sample_qc.n_transversion, 
#     mt.sample_qc.n_star, mt.sample_qc.r_ti_tv, mt.sample_qc.r_het_hom_var, mt.sample_qc.r_insertion_deletion)

# mt_1kg = mt_1kg.select_cols(mt_1kg.sample_qc.call_rate, mt_1kg.sample_qc.n_called, mt_1kg.sample_qc.n_not_called,
#     mt_1kg.sample_qc.n_hom_ref, mt_1kg.sample_qc.n_het, mt_1kg.sample_qc.n_hom_var, mt_1kg.sample_qc.n_non_ref, mt_1kg.sample_qc.n_singleton,
#     mt_1kg.sample_qc.n_snp, mt_1kg.sample_qc.n_insertion, mt_1kg.sample_qc.n_deletion, mt_1kg.sample_qc.n_transition, mt_1kg.sample_qc.n_transversion, 
#     mt_1kg.sample_qc.n_star, mt_1kg.sample_qc.r_ti_tv, mt_1kg.sample_qc.r_het_hom_var,mt_1kg.sample_qc.r_insertion_deletion)

# mt = mt.union_cols(mt_1kg)
# mt = mt.checkpoint('gs://ibd-exomes-gnomad-subset/QC_round2/6.export/PCA/1.ibd.kgp.mt')



Reading in 1KG data...
Merging datasets...


2023-04-11 23:08:49 Hail: INFO: wrote matrix table with 424628 rows and 202691 columns in 46657 partitions to gs://ibd-exomes-gnomad-subset/QC_round2/6.export/PCA/1.ibd.kgp.mt


In [8]:
mt = hl.read_matrix_table('gs://ibd-exomes-gnomad-subset/QC_round2/6.export/PCA/1.ibd.kgp.mt')
mt = mt.filter_rows(mt.vep.most_severe_consequence == "synonymous_variant")
mt.count()

(24071, 202691)

In [9]:
mt = hl.variant_qc(mt, name='variant_qc')
mt = mt.filter_rows(mt.variant_qc.call_rate > 0.99, keep = True)
mt.count()

(20905, 202691)

In [10]:
#LD PRUNE
pruned_variants = hl.methods.ld_prune(mt.GT, r2 = 0.1, bp_window_size = 100000, block_size=1024)
print("Pruning...")
mt = mt.filter_rows(hl.is_defined(pruned_variants[mt.row_key]))
mt.count()

2023-04-18 01:14:02 Hail: INFO: ld_prune: running local pruning stage with max queue size of 1324 variants
2023-04-18 01:16:03 Hail: INFO: wrote table with 17640 rows in 46657 partitions to /tmp/SoXyejdL2LpriQlQQNyA4V
    Total size: 2.19 MiB
    * Rows: 2.19 MiB
    * Globals: 11.00 B
    * Smallest partition: 0 rows (21.00 B)
    * Largest partition:  6 rows (291.00 B)
2023-04-18 01:24:16 Hail: INFO: Wrote all 3564 blocks of 17640 x 202691 matrix with block size 1024.
2023-04-18 01:30:54 Hail: INFO: wrote table with 5705 rows in 30 partitions to /tmp/cSYzCkHXrElK2odCggdpH3
    Total size: 371.62 KiB
    * Rows: 94.69 KiB
    * Globals: 276.93 KiB
    * Smallest partition: 0 rows (21.00 B)
    * Largest partition:  578 rows (7.96 KiB)


Pruning...


(14350, 202691)

In [11]:
#Pull out 1KG samples
print("Grabbing 1KG samples...")
kg_ht = hl.import_table('gs://twist-ibd/103.PCA/103.002.hail.kgp.pca/1kg_samples.tsv.bgz')
kg_ht = kg_ht.key_by('s')
kg_mt = mt.filter_cols(hl.is_defined(kg_ht[mt.s]), keep=True)

Grabbing 1KG samples...


2023-04-18 01:32:03 Hail: INFO: Reading table without type imputation
  Loading field 's' as type str (not specified)


In [12]:
#PCA on the 1kG
print("Performing PCA...")
eigenvalues, scores, loadings = hl.hwe_normalized_pca(kg_mt.GT, k=10, compute_loadings=True)
kg_mt = kg_mt.annotate_rows(pca_af=hl.agg.mean(kg_mt.GT.n_alt_alleles()) / 2)
loadings = loadings.annotate(pca_af=kg_mt.rows()[loadings.key].pca_af)

#Project loadings onto all samples
print("Projecting loadings onto all samples...")
projections = pc_project(mt, loadings)

print("Exporting results...")
projections.export('gs://ibd-exomes-gnomad-subset/QC_round2/6.export/PCA/ibd.pca.tsv.bgz')

Performing PCA...


2023-04-18 01:32:47 Hail: INFO: hwe_normalize: found 14350 variants after filtering out monomorphic sites.
2023-04-18 01:33:51 Hail: INFO: pca: running PCA with 10 components...
2023-04-18 01:54:43 Hail: INFO: Coerced sorted dataset


Projecting loadings onto all samples...
Exporting results...


2023-04-18 01:55:07 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'
2023-04-18 02:22:02 Hail: INFO: Coerced sorted dataset
2023-04-18 02:22:04 Hail: INFO: merging 17 files totalling 19.7M...
2023-04-18 02:22:05 Hail: INFO: while writing:
    gs://ibd-exomes-gnomad-subset/QC_round2/6.export/PCA/ibd.pca.tsv.bgz
  merge time: 500.481ms


In [13]:
gnomad_mt = mt.filter_cols(hl.is_defined(kg_ht[mt.s]), keep=False)
gnomad_mt.write('gs://ibd-exomes-gnomad-subset/QC_round2/6.export/PCA/2.gnomad.pre-pca.mt', overwrite=True)

2023-04-18 02:28:01 Hail: INFO: wrote matrix table with 14350 rows and 200187 columns in 46657 partitions to gs://ibd-exomes-gnomad-subset/QC_round2/6.export/PCA/2.gnomad.pre-pca.mt
